Завдання 1. Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери. Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv(r"C:\Users\user\Desktop\python\jup\data\customer_segmentation_train.csv")
df = df.drop(columns=["ID"])

num_cols = ["Age", "Work_Experience", "Family_Size"]
cat_cols = ["Gender", "Ever_Married", "Graduated", "Profession", "Var_1"]

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

df["Spending_Score"] = df["Spending_Score"].map({"Low": 0, "Average": 1, "High": 2})

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

target_encoder = LabelEncoder()
df["Segmentation"] = target_encoder.fit_transform(df["Segmentation"])

X = df.drop(columns=["Segmentation"])
y = df["Segmentation"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape

((6454, 9), (1614, 9))

Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

Переглянути інформацію про метод SMOTENC і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

Підказка: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу SMOTENC(..., categorical_features=cat_feature_indeces).

Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками SMOTEN

In [9]:
df = pd.read_csv(r"C:\Users\user\Desktop\python\jup\data\customer_segmentation_train.csv")
df = df.drop(columns=["ID"])

num_cols = ["Age", "Work_Experience", "Family_Size"]
cat_cols = ["Gender", "Ever_Married", "Graduated", "Profession", "Var_1"]

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

df["Spending_Score"] = df["Spending_Score"].map({"Low": 0, "Average": 1, "High": 2})

# категоріальні ознаки лишаємо як ОДИН цілочисельний стовпець кожна —
# це потрібно, щоб потім вказати SMOTENC, які саме колонки категоріальні
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df["Segmentation"] = LabelEncoder().fit_transform(df["Segmentation"])

X = df.drop(columns=["Segmentation"])
y = df["Segmentation"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE

continuous_cols = ["Age", "Work_Experience", "Family_Size", "Spending_Score"]
X_train_num = X_train[continuous_cols]

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_num, y_train)

y_train_smote.value_counts()

Segmentation
0    1814
1    1814
2    1814
3    1814
Name: count, dtype: int64

In [15]:
from imblearn.combine import SMOTETomek

smote_tomek = SMOTETomek(smote=SMOTENC(categorical_features=cat_feature_indices, random_state=42),
                          random_state=42)

X_train_smotetomek, y_train_smotetomek = smote_tomek.fit_resample(X_train, y_train)

y_train_smotetomek.value_counts()

Segmentation
3    1565
2    1559
1    1523
0    1487
Name: count, dtype: int64

Завдання 3
1.Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.
2.Виміряйте якість кожної з натренованих моделей використовуючи sklearn.metrics.classification_report.
3.Напишіть, яку метрику ви обрали для порівняння моделей.
4.Яка модель найкраща?
5.Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score

def train_evaluate(X_tr, y_tr, name):
    model = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_test)
    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=target_encoder.classes_, digits=3))
    return f1_score(y_test, y_pred, average="macro")

f1_orig = train_evaluate(X_train, y_train, "Оригінальні дані")
f1_smote = train_evaluate(X_train_smotenc, y_train_smotenc, "SMOTE (SMOTENC)")
f1_smotetomek = train_evaluate(X_train_smotetomek, y_train_smotetomek, "SMOTE-Tomek")

===== Оригінальні дані =====
              precision    recall  f1-score   support

           A      0.395     0.338     0.364       394
           B      0.424     0.105     0.168       372
           C      0.468     0.673     0.552       394
           D      0.583     0.795     0.673       454

    accuracy                          0.494      1614
   macro avg      0.467     0.478     0.439      1614
weighted avg      0.472     0.494     0.452      1614

===== SMOTE (SMOTENC) =====
              precision    recall  f1-score   support

           A      0.416     0.363     0.388       394
           B      0.374     0.183     0.245       372
           C      0.472     0.622     0.537       394
           D      0.612     0.767     0.680       454

    accuracy                          0.498      1614
   macro avg      0.468     0.484     0.463      1614
weighted avg      0.475     0.498     0.474      1614

===== SMOTE-Tomek =====
              precision    recall  f1-score   sup

4. Через те що в нас 4 класи, важливо аби модель добре працювала на кожному з них, тому MACRO F1-score, краще підходить, бо усереднює f1-score з однаковою вагою по класахю.

5. Різниця між SMOTE (0.461) і SMOTE-Tomek (0.463) мінімальна, бо Tomek-очищення прибрало лише невелику частину точок, це не змінює суттєво результат.